# Scenario 4 - Continuous Mountain Car: Linear Non-Null Action Cost

**Objective:** reach the flag while minimizing the number of non-null continuous actions.

**Base environment:** `MountainCarContinuous-v0`

**Adapted cost:** the default continuous environment penalizes `0.1 * action^2`. Scenario 4 changes this to a linear count cost:

- `0` if `abs(action) <= non_null_threshold`
- `-1` if `abs(action) > non_null_threshold`
- `+100` when the goal is reached

The important difference from Scenario 2 is that action intensity no longer changes the cost. A tiny non-null push and a full-strength push both spend one action unit. That makes the objective closer to Scenario 3, but with a continuous action space: the agent should learn sparse, well-timed pushes rather than continuous small-force control.


## Section 1 - Setup and Dependencies

This cell keeps the notebook self-contained. It defines the wrapper, agent builders, training callbacks, evaluation utilities, and plotting helpers directly here so no extra Scenario 4 `.py` file is required.

TensorBoard is enabled through a dedicated `runs/s04_tensorboard` directory. The previous `FileNotFoundError` came from a TensorBoard writer trying to write to `runs/s04_sac/...` after that directory had been removed while training was active. This notebook creates the log directory before SB3 starts; do not delete `runs/s04_tensorboard` while training is running.


In [ ]:
import os
import random
import warnings
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

warnings.filterwarnings('ignore')

SEED = 42
CHECKPOINT_DIR = Path('checkpoints')
RUN_DIR = Path('runs')
SCENARIO_PREFIX = 's04'
USE_TENSORBOARD = True
NON_NULL_THRESHOLD = 1e-3
TENSORBOARD_LOG_DIR = RUN_DIR / 's04_tensorboard'

CHECKPOINT_DIR.mkdir(exist_ok=True)
RUN_DIR.mkdir(exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

try:
    import stable_baselines3
    from stable_baselines3 import SAC, TD3
    from stable_baselines3.common.callbacks import BaseCallback
    from stable_baselines3.common.monitor import Monitor
    from stable_baselines3.common.noise import NormalActionNoise
    SB3_AVAILABLE = True
except ModuleNotFoundError as exc:
    stable_baselines3 = None
    SAC = TD3 = BaseCallback = Monitor = NormalActionNoise = None
    SB3_AVAILABLE = False
    SB3_IMPORT_ERROR = exc

print(f'Python environment ready. SB3 available: {SB3_AVAILABLE}')
if SB3_AVAILABLE:
    print(f'stable-baselines3 version: {stable_baselines3.__version__}')
else:
    print('Install missing training dependency with: %pip install stable-baselines3 tensorboard')


def require_sb3():
    if not SB3_AVAILABLE:
        raise ModuleNotFoundError(
            'stable_baselines3 is not installed in this notebook kernel. '
            'Run: %pip install stable-baselines3 tensorboard'
        ) from SB3_IMPORT_ERROR


def action_abs_magnitude(action):
    action_array = np.asarray(action, dtype=np.float32)
    if action_array.size == 0:
        return 0.0
    return float(np.max(np.abs(action_array)))


class LinearNonNullActionCostWrapper(gym.Wrapper):
    """Scenario 4 reward adaptation for MountainCarContinuous-v0."""

    def __init__(self, env, action_cost=1.0, non_null_threshold=NON_NULL_THRESHOLD, goal_bonus=100.0):
        super().__init__(env)
        self.action_cost = float(action_cost)
        self.non_null_threshold = float(non_null_threshold)
        self.goal_bonus = float(goal_bonus)
        self.episode_non_null_actions = 0
        self.episode_linear_action_cost = 0.0
        self.episode_steps = 0

    def reset(self, **kwargs):
        self.episode_non_null_actions = 0
        self.episode_linear_action_cost = 0.0
        self.episode_steps = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, raw_reward, terminated, truncated, info = self.env.step(action)
        action_magnitude = action_abs_magnitude(action)
        non_null_action = action_magnitude > self.non_null_threshold
        linear_action_cost = self.action_cost if non_null_action else 0.0
        goal_bonus = self.goal_bonus if terminated else 0.0
        reward = goal_bonus - linear_action_cost

        self.episode_steps += 1
        if non_null_action:
            self.episode_non_null_actions += 1
        self.episode_linear_action_cost += linear_action_cost

        info = dict(info)
        info.update({
            'raw_reward': float(raw_reward),
            'action_magnitude': action_magnitude,
            'non_null_action': non_null_action,
            'linear_action_cost': linear_action_cost,
            'goal_bonus': goal_bonus,
            'episode_non_null_actions': self.episode_non_null_actions,
            'episode_linear_action_cost': self.episode_linear_action_cost,
            'episode_steps': self.episode_steps,
        })
        return obs, reward, terminated, truncated, info


def make_scenario4_env(render_mode=None, action_cost=1.0, non_null_threshold=NON_NULL_THRESHOLD, goal_bonus=100.0):
    env = gym.make('MountainCarContinuous-v0', render_mode=render_mode)
    return LinearNonNullActionCostWrapper(
        env,
        action_cost=action_cost,
        non_null_threshold=non_null_threshold,
        goal_bonus=goal_bonus,
    )


def tensorboard_log_dir():
    if not USE_TENSORBOARD:
        return None
    TENSORBOARD_LOG_DIR.mkdir(parents=True, exist_ok=True)
    return str(TENSORBOARD_LOG_DIR)


if SB3_AVAILABLE:
    class EpisodeMetricsCallback(BaseCallback):
        """Stores per-episode metrics from Monitor and wrapper info."""

        def __init__(self):
            super().__init__()
            self.episode_rewards = []
            self.episode_lengths = []
            self.non_null_counts = []
            self.linear_costs = []
            self.successes = []

        def _on_step(self):
            infos = self.locals.get('infos', [])
            dones = self.locals.get('dones', [])
            for info, done in zip(infos, dones):
                if not done:
                    continue
                episode_info = info.get('episode', {})
                self.episode_rewards.append(float(episode_info.get('r', np.nan)))
                self.episode_lengths.append(int(episode_info.get('l', info.get('episode_steps', 0))))
                self.non_null_counts.append(int(info.get('episode_non_null_actions', 0)))
                self.linear_costs.append(float(info.get('episode_linear_action_cost', 0.0)))
                self.successes.append(bool(info.get('goal_bonus', 0.0) > 0.0))
            return True

        def as_dict(self):
            return {
                'episode_rewards': np.array(self.episode_rewards, dtype=float),
                'episode_lengths': np.array(self.episode_lengths, dtype=float),
                'non_null_counts': np.array(self.non_null_counts, dtype=float),
                'linear_costs': np.array(self.linear_costs, dtype=float),
                'successes': np.array(self.successes, dtype=float),
            }


    class SuccessEvalCallback(BaseCallback):
        """Periodic evaluation; saves best model by success rate, then lower action count."""

        def __init__(self, eval_env, label, eval_freq=5000, n_eval_episodes=10):
            super().__init__()
            self.eval_env = eval_env
            self.label = label
            self.eval_freq = int(eval_freq)
            self.n_eval_episodes = int(n_eval_episodes)
            self.best_success_rate = -1.0
            self.best_mean_non_null = np.inf
            self.best_mean_reward = -np.inf
            self.history = []
            self.save_path = CHECKPOINT_DIR / f'{SCENARIO_PREFIX}_{label.lower()}_best'
            self.save_path.mkdir(exist_ok=True)

        def _on_step(self):
            if self.eval_freq <= 0 or self.n_calls % self.eval_freq != 0:
                return True
            metrics = evaluate_model(self.model, self.eval_env, n_episodes=self.n_eval_episodes, seed=SEED + 20000)
            row = {
                'timesteps': self.num_timesteps,
                'mean_reward': metrics['mean_reward'],
                'success_rate': metrics['success_rate'],
                'mean_non_null_actions': metrics['mean_non_null_actions'],
            }
            self.history.append(row)
            print(
                f"[{self.label} eval] steps={self.num_timesteps} "
                f"reward={row['mean_reward']:.2f} success={100*row['success_rate']:.0f}% "
                f"non_null={row['mean_non_null_actions']:.1f}"
            )
            is_better = (
                row['success_rate'] > self.best_success_rate
                or (
                    np.isclose(row['success_rate'], self.best_success_rate)
                    and row['mean_non_null_actions'] < self.best_mean_non_null
                )
                or (
                    np.isclose(row['success_rate'], self.best_success_rate)
                    and np.isclose(row['mean_non_null_actions'], self.best_mean_non_null)
                    and row['mean_reward'] > self.best_mean_reward
                )
            )
            if is_better:
                self.best_success_rate = row['success_rate']
                self.best_mean_non_null = row['mean_non_null_actions']
                self.best_mean_reward = row['mean_reward']
                self.model.save(self.save_path / 'best_model')
            return True

else:
    EpisodeMetricsCallback = None
    SuccessEvalCallback = None


def build_sac_model(env=None, seed=SEED):
    require_sb3()
    env = Monitor(env or make_scenario4_env())
    return SAC(
        'MlpPolicy',
        env,
        learning_rate=3e-4,
        buffer_size=50000,
        learning_starts=0,
        batch_size=512,
        tau=0.01,
        gamma=0.9999,
        train_freq=32,
        gradient_steps=32,
        ent_coef=0.1,
        use_sde=True,
        verbose=0,
        tensorboard_log=tensorboard_log_dir(),
        policy_kwargs=dict(log_std_init=-3.67, net_arch=[64, 64]),
        seed=seed,
    )


def build_td3_model(env=None, seed=SEED):
    require_sb3()
    env = Monitor(env or make_scenario4_env())
    action_dim = env.action_space.shape[-1]
    noise = NormalActionNoise(mean=np.zeros(action_dim), sigma=0.15 * np.ones(action_dim))
    return TD3(
        'MlpPolicy',
        env,
        learning_rate=3e-4,
        buffer_size=50000,
        learning_starts=1000,
        batch_size=256,
        tau=0.01,
        gamma=0.9999,
        train_freq=(1, 'step'),
        gradient_steps=1,
        action_noise=noise,
        policy_delay=2,
        verbose=0,
        tensorboard_log=tensorboard_log_dir(),
        policy_kwargs=dict(net_arch=[64, 64]),
        seed=seed,
    )


AGENT_BUILDERS = {
    'SAC': build_sac_model,
    'TD3': build_td3_model,
}


def evaluate_action_sequence(env, actions, seed=None):
    state, _ = env.reset(seed=seed) if seed is not None else env.reset()
    total_reward = 0.0
    non_null_actions = 0
    linear_action_cost = 0.0
    action_magnitudes = []
    steps = 0
    success = False

    for action in actions:
        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        non_null_actions = info.get('episode_non_null_actions', non_null_actions)
        linear_action_cost = info.get('episode_linear_action_cost', linear_action_cost)
        action_magnitudes.append(info.get('action_magnitude', action_abs_magnitude(action)))
        steps += 1
        success = success or terminated
        if terminated or truncated:
            break

    return {
        'total_reward': total_reward,
        'non_null_actions': non_null_actions,
        'linear_action_cost': linear_action_cost,
        'mean_action_magnitude': float(np.mean(action_magnitudes)) if action_magnitudes else 0.0,
        'steps': steps,
        'success': success,
        'final_state': np.asarray(state, dtype=np.float32),
    }


def rollout_policy(env, policy_fn, seed=1000, max_steps=999):
    state, _ = env.reset(seed=seed)
    states = [state.copy()]
    actions = []
    rewards = []
    infos = []
    done = False
    steps = 0

    while not done and steps < max_steps:
        action = np.asarray(policy_fn(state), dtype=np.float32).reshape(env.action_space.shape)
        state, reward, terminated, truncated, info = env.step(action)
        states.append(state.copy())
        actions.append(action.copy())
        rewards.append(float(reward))
        infos.append(info)
        done = terminated or truncated
        steps += 1

    return {
        'states': np.array(states),
        'actions': np.array(actions).reshape(-1),
        'rewards': np.array(rewards, dtype=float),
        'infos': infos,
        'total_reward': float(np.sum(rewards)),
        'success': bool(infos[-1].get('goal_bonus', 0.0) > 0.0) if infos else False,
        'non_null_actions': int(infos[-1].get('episode_non_null_actions', 0)) if infos else 0,
        'linear_action_cost': float(infos[-1].get('episode_linear_action_cost', 0.0)) if infos else 0.0,
        'steps': steps,
    }


def evaluate_model(model, env=None, n_episodes=100, seed=1000):
    env = env or make_scenario4_env()
    episode_rows = []
    for ep in range(n_episodes):
        traj = rollout_policy(env, lambda s: model.predict(s, deterministic=True)[0], seed=seed + ep)
        episode_rows.append(traj)

    rewards = np.array([r['total_reward'] for r in episode_rows], dtype=float)
    successes = np.array([r['success'] for r in episode_rows], dtype=float)
    steps = np.array([r['steps'] for r in episode_rows], dtype=float)
    non_null = np.array([r['non_null_actions'] for r in episode_rows], dtype=float)
    linear_costs = np.array([r['linear_action_cost'] for r in episode_rows], dtype=float)
    mean_abs_actions = np.array([
        float(np.mean(np.abs(r['actions']))) if len(r['actions']) else 0.0
        for r in episode_rows
    ])

    return {
        'episodes': episode_rows,
        'rewards': rewards,
        'mean_reward': float(np.mean(rewards)) if len(rewards) else 0.0,
        'std_reward': float(np.std(rewards)) if len(rewards) else 0.0,
        'success_rate': float(np.mean(successes)) if len(successes) else 0.0,
        'steps': steps,
        'mean_steps': float(np.mean(steps)) if len(steps) else 0.0,
        'non_null_actions': non_null,
        'mean_non_null_actions': float(np.mean(non_null)) if len(non_null) else 0.0,
        'linear_action_costs': linear_costs,
        'mean_linear_action_cost': float(np.mean(linear_costs)) if len(linear_costs) else 0.0,
        'mean_abs_actions': mean_abs_actions,
        'mean_abs_action': float(np.mean(mean_abs_actions)) if len(mean_abs_actions) else 0.0,
    }


def evaluate_random_policy(env=None, n_episodes=100, seed=1000):
    env = env or make_scenario4_env()
    episode_rows = []
    rng = np.random.default_rng(seed)
    for ep in range(n_episodes):
        traj = rollout_policy(
            env,
            lambda s: rng.uniform(env.action_space.low, env.action_space.high),
            seed=seed + ep,
        )
        episode_rows.append(traj)

    rewards = np.array([r['total_reward'] for r in episode_rows], dtype=float)
    successes = np.array([r['success'] for r in episode_rows], dtype=float)
    non_null = np.array([r['non_null_actions'] for r in episode_rows], dtype=float)
    steps = np.array([r['steps'] for r in episode_rows], dtype=float)
    return {
        'episodes': episode_rows,
        'rewards': rewards,
        'mean_reward': float(np.mean(rewards)),
        'std_reward': float(np.std(rewards)),
        'success_rate': float(np.mean(successes)),
        'non_null_actions': non_null,
        'mean_non_null_actions': float(np.mean(non_null)),
        'steps': steps,
        'mean_steps': float(np.mean(steps)),
        'mean_linear_action_cost': float(np.mean(non_null)),
        'mean_abs_action': float(np.mean([np.mean(np.abs(r['actions'])) for r in episode_rows])),
    }


def train_agent(label, total_timesteps=50000, seed=SEED, eval_freq=5000, n_eval_episodes=10):
    require_sb3()
    builder = AGENT_BUILDERS[label]
    model = builder(seed=seed)
    train_callback = EpisodeMetricsCallback()
    eval_callback = SuccessEvalCallback(make_scenario4_env(), label, eval_freq=eval_freq, n_eval_episodes=n_eval_episodes)
    model.learn(
        total_timesteps=total_timesteps,
        callback=[train_callback, eval_callback],
        log_interval=50,
        tb_log_name=f'{SCENARIO_PREFIX}_{label.lower()}',
    )
    model.save(CHECKPOINT_DIR / f'{SCENARIO_PREFIX}_{label.lower()}_last')
    return model, train_callback.as_dict(), eval_callback.history


def moving_average(values, window=50):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return np.array([]), np.array([])
    ma = np.convolve(values, np.ones(window) / window, mode='valid')
    return np.arange(window - 1, len(values)), ma


def state_grid(env, n_grid=60):
    pos_low, vel_low = env.observation_space.low
    pos_high, vel_high = env.observation_space.high
    pos = np.linspace(pos_low, pos_high, n_grid)
    vel = np.linspace(vel_low, vel_high, n_grid)
    return pos, vel


def model_action(model, state):
    return float(np.asarray(model.predict(state, deterministic=True)[0]).reshape(-1)[0])


def print_metric_table(results):
    print(f"{'Agent':<10} {'Mean reward':>13} {'Success':>10} {'Non-null':>12} {'Steps':>10} {'|action|':>10}")
    print('-' * 70)
    for name, metrics in results.items():
        print(
            f"{name:<10} "
            f"{metrics['mean_reward']:>13.2f} "
            f"{100 * metrics['success_rate']:>9.1f}% "
            f"{metrics['mean_non_null_actions']:>12.1f} "
            f"{metrics['mean_steps']:>10.1f} "
            f"{metrics.get('mean_abs_action', 0.0):>10.3f}"
        )


print('Scenario 4 setup complete.')


## Section 2 - Environment and Reward Adaptation

The state is still the two-dimensional Mountain Car state `(position, velocity)`. The action is a continuous one-dimensional force in `[-1, 1]`.

The custom wrapper is the core Scenario 4 adaptation: it ignores the built-in quadratic reward and returns a count-based cost instead. `NON_NULL_THRESHOLD` is only a tiny numerical tolerance around zero, so the adapted reward represents the assignment objective without creating a free-force dead zone. This makes the performance objective easy to audit because the engineered reward is exactly:

`episode reward = 100 * success - number_of_non_null_actions`


In [ ]:
raw_env = gym.make('MountainCarContinuous-v0')
wrapped_env = make_scenario4_env(non_null_threshold=NON_NULL_THRESHOLD)
raw_env.reset(seed=SEED)
wrapped_env.reset(seed=SEED)

print('=== MountainCarContinuous-v0 / Scenario 4 ===')
print(f'Observation space: {raw_env.observation_space}')
print(f'Action space: {raw_env.action_space}')
print('Original reward: -0.1 * action^2 + 100 at goal')
print('Scenario 4 reward: -1 per non-null action + 100 at goal')

print('\nReward sanity check:')
state, _ = wrapped_env.reset(seed=SEED)
for action in [
    np.array([0.0], dtype=np.float32),
    np.array([NON_NULL_THRESHOLD / 2], dtype=np.float32),
    np.array([NON_NULL_THRESHOLD * 2], dtype=np.float32),
    np.array([1.0], dtype=np.float32),
]:
    _, reward, terminated, truncated, info = wrapped_env.step(action)
    print(
        f'action={float(action[0]):+.4f}  reward={reward:+.1f}  '
        f'non_null={info["non_null_action"]}  '
        f'episode_non_null={info["episode_non_null_actions"]}'
    )
    if terminated or truncated:
        break

raw_env.close()
wrapped_env.close()


## Section 3 - Baseline Testbed

A random policy is a useful lower bound. With a threshold near zero, almost every random continuous action is non-null, so random behavior should accumulate a very large action count and low reward.


In [ ]:
random_eval = evaluate_random_policy(make_scenario4_env(), n_episodes=20, seed=SEED)
print_metric_table({'Random': random_eval})


## Section 4 - Algorithms

Scenario 4 has a continuous action space, so a tabular Q-table is not appropriate unless we discretize actions and lose the point of the scenario. We compare two continuous-control deep RL agents:

- **SAC (Soft Actor-Critic):** stochastic actor-critic, robust exploration, strong default for continuous control.
- **TD3 (Twin Delayed DDPG):** deterministic actor-critic, useful comparison because it often learns sharper force decisions.

Both agents optimize the same adapted environment and are evaluated with the same objective metrics: success rate, mean reward, mean steps, and mean number of non-null actions.


In [ ]:
TRAIN_TIMESTEPS = 50000
EVAL_FREQ = 5000
N_EVAL_EPISODES_DURING_TRAINING = 10
AGENTS_TO_TRAIN = ['SAC', 'TD3']

print(f'Agents: {AGENTS_TO_TRAIN}')
print(f'Train timesteps per agent: {TRAIN_TIMESTEPS:,}')
print(f'TensorBoard enabled: {USE_TENSORBOARD}')
if USE_TENSORBOARD:
    print(f'TensorBoard logs: {TENSORBOARD_LOG_DIR}')


## Section 5 - Training

This is the long-running cell. It trains both SAC and TD3 and stores compact histories for later plots. If you want a quick smoke test, temporarily reduce `TRAIN_TIMESTEPS` above to something like `2000`.


In [ ]:
models = {}
training_histories = {}
eval_histories = {}

for agent_name in AGENTS_TO_TRAIN:
    print(f'\nTraining {agent_name} on Scenario 4...')
    model, train_history, eval_history = train_agent(
        agent_name,
        total_timesteps=TRAIN_TIMESTEPS,
        seed=SEED,
        eval_freq=EVAL_FREQ,
        n_eval_episodes=N_EVAL_EPISODES_DURING_TRAINING,
    )
    models[agent_name] = model
    training_histories[agent_name] = train_history
    eval_histories[agent_name] = eval_history
    print(f'{agent_name} training complete. Episodes observed: {len(train_history["episode_rewards"])}')

print('\nAll Scenario 4 training complete.')


## Section 6 - Training Curves and Monitoring

The training curves monitor the engineered reward and the true objective metric. Reward should improve when the agent both reaches the goal and uses fewer non-null actions.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
window = 30

for name, history in training_histories.items():
    rewards = history['episode_rewards']
    non_null = history['non_null_counts']
    axes[0, 0].plot(rewards, alpha=0.25, label=f'{name} reward')
    x, ma = moving_average(rewards, window=window)
    if len(ma):
        axes[0, 0].plot(x, ma, lw=2, label=f'{name} reward MA')

    axes[0, 1].plot(non_null, alpha=0.25, label=f'{name} non-null')
    x, ma = moving_average(non_null, window=window)
    if len(ma):
        axes[0, 1].plot(x, ma, lw=2, label=f'{name} non-null MA')

for name, history in eval_histories.items():
    if not history:
        continue
    steps = [row['timesteps'] for row in history]
    success = [100 * row['success_rate'] for row in history]
    non_null = [row['mean_non_null_actions'] for row in history]
    axes[1, 0].plot(steps, success, marker='o', label=name)
    axes[1, 1].plot(steps, non_null, marker='o', label=name)

axes[0, 0].set_title('Episode reward during training')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Reward')
axes[0, 1].set_title('Non-null actions per episode')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Count')
axes[1, 0].set_title('Periodic evaluation success rate')
axes[1, 0].set_xlabel('Timesteps')
axes[1, 0].set_ylabel('Success rate (%)')
axes[1, 1].set_title('Periodic evaluation non-null action count')
axes[1, 1].set_xlabel('Timesteps')
axes[1, 1].set_ylabel('Mean count')

for ax in axes.flat:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 's04_training_curves.png', dpi=150)
plt.show()


## Section 7 - Evaluation

Evaluation separates the engineered reward from the objective behavior. For Scenario 4, the most important objective metric is the number of non-null actions used to reach the goal.


In [ ]:
evaluation_results = {'Random': random_eval}
for name, model in models.items():
    evaluation_results[name] = evaluate_model(model, make_scenario4_env(), n_episodes=100, seed=1000)

print_metric_table(evaluation_results)


In [ ]:
metric_names = ['mean_reward', 'success_rate', 'mean_non_null_actions', 'mean_steps']
metric_titles = ['Mean reward', 'Success rate', 'Mean non-null actions', 'Mean steps']
agent_names = list(evaluation_results.keys())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric, title in zip(axes, metric_names, metric_titles):
    values = [evaluation_results[name][metric] for name in agent_names]
    if metric == 'success_rate':
        values = [100 * v for v in values]
    ax.bar(agent_names, values, color=['gray', 'steelblue', 'coral'][:len(agent_names)])
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)
    if metric == 'success_rate':
        ax.set_ylabel('%')
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 's04_evaluation_comparison.png', dpi=150)
plt.show()


## Section 8 - Policy Heatmaps

The heatmaps show what each learned policy does across the `(position, velocity)` state space. A good Scenario 4 policy should concentrate useful non-null actions where thrust increases mechanical energy and avoid wasting action count where possible.


In [ ]:
def plot_action_heatmaps(models, threshold=NON_NULL_THRESHOLD, n_grid=60):
    env = make_scenario4_env(non_null_threshold=threshold)
    pos, vel = state_grid(env, n_grid=n_grid)
    fig, axes = plt.subplots(len(models), 2, figsize=(12, 4 * len(models)), squeeze=False)

    for row, (name, model) in enumerate(models.items()):
        action_grid = np.zeros((n_grid, n_grid))
        cost_grid = np.zeros((n_grid, n_grid))
        for i, p in enumerate(pos):
            for j, v in enumerate(vel):
                a = model_action(model, np.array([p, v], dtype=np.float32))
                action_grid[j, i] = a
                cost_grid[j, i] = float(abs(a) > threshold)

        im0 = axes[row, 0].imshow(
            action_grid,
            extent=[pos[0], pos[-1], vel[0], vel[-1]],
            origin='lower',
            aspect='auto',
            cmap='coolwarm',
            vmin=-1,
            vmax=1,
        )
        plt.colorbar(im0, ax=axes[row, 0], label='Continuous action')
        axes[row, 0].set_title(f'{name}: action field')

        im1 = axes[row, 1].imshow(
            cost_grid,
            extent=[pos[0], pos[-1], vel[0], vel[-1]],
            origin='lower',
            aspect='auto',
            cmap='Greys',
            vmin=0,
            vmax=1,
        )
        plt.colorbar(im1, ax=axes[row, 1], label='Non-null cost indicator')
        axes[row, 1].set_title(f'{name}: where actions cost fuel')

        for ax in axes[row]:
            ax.axvline(0.45, color='gold', ls='--', lw=2, label='Goal')
            ax.set_xlabel('Position')
            ax.set_ylabel('Velocity')
            ax.legend(loc='upper left')

    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / 's04_policy_heatmaps.png', dpi=150)
    plt.show()
    env.close()

plot_action_heatmaps(models)


## Section 9 - Phase Portraits and State Visitation

The phase portrait shows the trajectories in the physical state space. The state-visitation heatmap shows where each policy spends its time while executing the learned strategy.


In [ ]:
def collect_model_trajectories(model, n_episodes=30, seed=3000):
    env = make_scenario4_env()
    trajectories = []
    for ep in range(n_episodes):
        traj = rollout_policy(env, lambda s: model.predict(s, deterministic=True)[0], seed=seed + ep)
        trajectories.append(traj)
    env.close()
    return trajectories

trajectory_sets = {name: collect_model_trajectories(model) for name, model in models.items()}

fig, axes = plt.subplots(1, len(trajectory_sets), figsize=(7 * len(trajectory_sets), 5), squeeze=False)
for ax, (name, trajs) in zip(axes.flat, trajectory_sets.items()):
    rewards = [t['total_reward'] for t in trajs]
    vmin, vmax = min(rewards), max(rewards)
    cmap = plt.cm.viridis
    for traj in trajs:
        states = traj['states']
        color = cmap((traj['total_reward'] - vmin) / max(vmax - vmin, 1e-8))
        ax.plot(states[:, 0], states[:, 1], alpha=0.45, lw=0.9, color=color)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Episode reward')
    ax.axvline(0.45, color='gold', ls='--', lw=2, label='Goal')
    ax.set_title(f'{name}: phase portrait')
    ax.set_xlabel('Position')
    ax.set_ylabel('Velocity')
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 's04_phase_portraits.png', dpi=150)
plt.show()


In [ ]:
def build_visit_grid_from_trajectories(trajs, env, n_grid=50):
    pos_low, vel_low = env.observation_space.low
    pos_high, vel_high = env.observation_space.high
    grid = np.zeros((n_grid, n_grid))
    for traj in trajs:
        for s in traj['states']:
            pi = int(np.clip((s[0] - pos_low) / (pos_high - pos_low) * (n_grid - 1), 0, n_grid - 1))
            vi = int(np.clip((s[1] - vel_low) / (vel_high - vel_low) * (n_grid - 1), 0, n_grid - 1))
            grid[pi, vi] += 1
    return grid

env_for_grid = make_scenario4_env()
fig, axes = plt.subplots(1, len(trajectory_sets), figsize=(7 * len(trajectory_sets), 5), squeeze=False)
for ax, (name, trajs) in zip(axes.flat, trajectory_sets.items()):
    grid = build_visit_grid_from_trajectories(trajs, env_for_grid)
    pos_low, vel_low = env_for_grid.observation_space.low
    pos_high, vel_high = env_for_grid.observation_space.high
    im = ax.imshow(np.log1p(grid).T, extent=[pos_low, pos_high, vel_low, vel_high], origin='lower', cmap='hot', aspect='auto')
    plt.colorbar(im, ax=ax, label='log(1 + visits)')
    ax.axvline(0.45, color='cyan', ls='--', lw=2, label='Goal')
    ax.set_title(f'{name}: state visitation')
    ax.set_xlabel('Position')
    ax.set_ylabel('Velocity')
    ax.legend()
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 's04_state_visitation.png', dpi=150)
plt.show()
env_for_grid.close()


## Section 10 - Action-Time Analysis

This plot makes the cost function concrete: every non-null action has the same cost, regardless of its magnitude. The agent gets no discount for tiny non-zero actions, so useful policies should avoid jitter around zero.


In [ ]:
fig, axes = plt.subplots(len(models), 2, figsize=(14, 4 * len(models)), squeeze=False)
for row, (name, model) in enumerate(models.items()):
    env = make_scenario4_env()
    traj = rollout_policy(env, lambda s: model.predict(s, deterministic=True)[0], seed=5000)
    env.close()
    actions = traj['actions']
    rewards = traj['rewards']
    non_null = np.abs(actions) > NON_NULL_THRESHOLD
    t = np.arange(len(actions))

    axes[row, 0].plot(t, actions, color='steelblue', lw=1.5, label='Action')
    axes[row, 0].fill_between(t, -1, 1, where=non_null, color='red', alpha=0.12, label='Cost paid')
    axes[row, 0].axhline(0, color='black', lw=0.8)
    axes[row, 0].set_title(f'{name}: action over time')
    axes[row, 0].set_xlabel('Step')
    axes[row, 0].set_ylabel('Force')
    axes[row, 0].legend()
    axes[row, 0].grid(alpha=0.3)

    axes[row, 1].plot(np.cumsum(rewards), color='darkgreen', lw=1.5, label='Cumulative reward')
    axes[row, 1].plot(np.cumsum(non_null), color='darkred', lw=1.5, label='Cumulative non-null actions')
    axes[row, 1].set_title(f'{name}: reward vs objective cost')
    axes[row, 1].set_xlabel('Step')
    axes[row, 1].legend()
    axes[row, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 's04_action_time_analysis.png', dpi=150)
plt.show()


## Section 11 - Policy Explanation With Simple Models

A small decision tree is used as an explanation tool, not as the RL policy. It approximates where the neural policy chooses null vs non-null actions and which state variable is most important.


In [ ]:
def explain_policy(model, name, n_samples=8000, threshold=NON_NULL_THRESHOLD):
    env = make_scenario4_env()
    rng = np.random.default_rng(SEED)
    low = env.observation_space.low
    high = env.observation_space.high
    X = rng.uniform(low, high, size=(n_samples, 2)).astype(np.float32)
    actions = np.array([model_action(model, s) for s in X])
    y_non_null = (np.abs(actions) > threshold).astype(int)

    classifier = DecisionTreeClassifier(max_depth=4, random_state=SEED)
    classifier.fit(X, y_non_null)

    regressor = DecisionTreeRegressor(max_depth=4, random_state=SEED)
    regressor.fit(X, actions)

    print(f'\n{name} non-null classifier tree:')
    print(export_text(classifier, feature_names=['position', 'velocity']))
    print(f'{name} action regressor feature importances:')
    print(f'  position: {regressor.feature_importances_[0]:.3f}')
    print(f'  velocity: {regressor.feature_importances_[1]:.3f}')
    env.close()
    return classifier, regressor

explainers = {name: explain_policy(model, name) for name, model in models.items()}

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(models))
width = 0.35
position_importance = [explainers[name][1].feature_importances_[0] for name in models]
velocity_importance = [explainers[name][1].feature_importances_[1] for name in models]
ax.bar(x - width/2, position_importance, width, label='Position')
ax.bar(x + width/2, velocity_importance, width, label='Velocity')
ax.set_xticks(x)
ax.set_xticklabels(list(models.keys()))
ax.set_ylim(0, 1)
ax.set_title('Feature importance for continuous action prediction')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 's04_feature_importance.png', dpi=150)
plt.show()


## Section 12 - Results Interpretation and Conclusions

This section turns the numeric outputs into the final claims for the report. It is generated from `evaluation_results`, so rerunning training updates the interpretation automatically.


In [ ]:
def summarize_scenario4_results(results):
    learned_agents = {name: metrics for name, metrics in results.items() if name != 'Random'}
    successful_agents = {
        name: metrics
        for name, metrics in learned_agents.items()
        if metrics['success_rate'] >= 0.8
    }

    print('Scenario 4 output interpretation')
    print('-' * 40)
    if successful_agents:
        best_name, best_metrics = min(
            successful_agents.items(),
            key=lambda item: item[1]['mean_non_null_actions'],
        )
        print(
            f'Best successful agent: {best_name} '
            f'({100 * best_metrics["success_rate"]:.0f}% success, '
            f'{best_metrics["mean_non_null_actions"]:.1f} non-null actions, '
            f'mean reward {best_metrics["mean_reward"]:.2f}).'
        )
    else:
        print('No learned agent reached the 80% success threshold; train longer or tune hyperparameters.')

    for name, metrics in learned_agents.items():
        if metrics['success_rate'] < 0.8:
            print(
                f'{name} is useful as a negative comparison here: '
                f'{100 * metrics["success_rate"]:.0f}% success and '
                f'{metrics["mean_non_null_actions"]:.1f} non-null actions.'
            )

    random_metrics = results.get('Random')
    if random_metrics and successful_agents:
        improvement = random_metrics['mean_non_null_actions'] - best_metrics['mean_non_null_actions']
        print(
            f'Compared with random exploration, {best_name} uses about '
            f'{improvement:.1f} fewer non-null actions per episode.'
        )

    print('\\nReport-ready conclusion:')
    print(
        'Scenario 4 is solved when an agent reaches the goal reliably while reducing the count '
        'of non-null continuous actions.'
    )
    if successful_agents:
        print(
            f'In the current run, {best_name} satisfies this objective; unsuccessful agents '
            'remain useful as negative comparisons under the same training budget.'
        )
    else:
        print(
            'In the current run, no learned agent satisfies the objective yet. Increase training, '
            'adjust exploration, or modify the success bonus/cost scale.'
        )


summarize_scenario4_results(evaluation_results)


### Comparison Notes for the Final Report

- **Versus Scenario 2:** Scenario 2 rewards smaller force magnitudes because cost is quadratic. Scenario 4 does not care about force magnitude once the action is non-null; the current SAC result reflects this by using strong actions but far fewer timesteps/non-null actions than random exploration.
- **Versus Scenario 3:** Scenario 3 and Scenario 4 both pay per active thrust, but Scenario 4 keeps a continuous force value. The policy analysis should focus on where the agent chooses to spend a non-null action, not on whether the force is small.
- **Objective vs reward:** the engineered reward is aligned with the objective because `reward = 100 * success - non_null_action_count`. Therefore, high reward only matters if success is high; a failed policy with low action count would not be acceptable.
- **Model comparison:** SAC is expected to be the stronger model here because its entropy-driven exploration handles the sparse success bonus better. TD3 is still useful in the notebook because it documents that not every continuous-control agent handles this count reward equally well.
